# Microglia Morphology Analysis

This notebook takes the results from the [microglia morphology quantification tool](https://github.com/isdneuroimaging/mmqt/tree/master) - pipeline published ["Automated Morphological Analysis of Microglia After Stroke"](https://www.frontiersin.org/journals/cellular-neuroscience/articles/10.3389/fncel.2018.00106/full).

Checked and made sure the following inclusion/exclusion criteria were used:
* thrDistBorderXY = 15; %15
* thrDistBorderZ  = 5; %8
* thrDistCells = 15; %15

### Generate a big dataframe with results from each experiment & respective series

Each series corresponds to a z stack from a region in the brain:
* Series 1: core
* Series 2: 300-600um
* Series 3: 600-900um
* Series 4: contralateral

In [ ]:
import os
import numpy as np
import pandas as pd
import re
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statannotations.Annotator import Annotator
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def extract_metadata_from_filename(filename):
    """
    Extract group, mouse_id, section_number, and series_number from filename.
    
    Example filenames: 
    - C57_aged_female_025910_S1_middle.nd2__series_4__features_summarized.txt
    - CTRL_NSG_young_male_C5M2_S5_middle_3.nd2__series_3_
    - CTRL_C57_young_male_025048_S1_end.nd2__series_2
    
    Returns:
        dict with keys: group, mouse_id, section_number, series_number, position
    """
    # Remove the file extension if present
    filename = filename.replace('.txt', '').rstrip('_')
    
    # Try pattern 1: 6-digit mouse ID format
    # This now captures everything before _XXXXXX_SX as the group
    pattern1 = r'(.+?)_(\d{6})_(S\d+)_(\w+)\.nd2__series_(\d+)(?:__features_summarized)?'
    match = re.search(pattern1, filename)
    
    if match:
        return {
            'group': match.group(1),
            'mouse_id': match.group(2),
            'section_number': match.group(3),
            'position': match.group(4),
            'series_number': match.group(5)
        }
    
    # Try pattern 2: Alphanumeric mouse ID format (e.g., C5M2)
    # Matches: GROUP_MOUSEID_SECTION_POSITION_NUMBER.nd2__series_N
    pattern2 = r'(.+?)_([A-Z]\d+[A-Z]\d+)_(S\d+)_(\w+)(?:_\d+)?\.nd2__series_(\d+)'
    match = re.search(pattern2, filename)
    
    if match:
        return {
            'group': match.group(1),
            'mouse_id': match.group(2),
            'section_number': match.group(3),
            'position': match.group(4),
            'series_number': match.group(5)
        }
    
    # If no pattern matches, return None values
    return {
        'group': None,
        'mouse_id': None,
        'section_number': None,
        'position': None,
        'series_number': None
    }
    
    # If no pattern matches, return None values
    return {
        'group': None,
        'mouse_id': None,
        'section_number': None,
        'position': None,
        'series_number': None
    }

In [ ]:
def find_all_txt_files(root_folder):
    """Find all features_of_cells.txt files within all subfolders"""
    txt_files = glob.glob(f"{root_folder}/**/*__series_*__features_summarized.txt", recursive=True)
    return txt_files

In [ ]:
def find_and_load_features_files(root_folder):
    """
    Search ALL subfolders recursively for files ending with 'features_summarized.txt'
    and combine them into a single DataFrame with metadata.
    
    Args:
        root_folder: Path to the root folder containing subfolders
        
    Returns:
        pandas DataFrame with all combined data and metadata
    """
    all_data = []
    files_found = 0
    files_loaded = 0
    
    print(f"Searching recursively for '*features_summarized.txt' files in: {root_folder}")
    print("=" * 80)
    
    # Get all subdirectories
    root_path = Path(root_folder)
    all_subdirs = [x for x in root_path.rglob('*') if x.is_dir()]
    
    print(f"Found {len(all_subdirs)} subdirectories to search")
    print("-" * 80)
    
    # Find all txt files recursively
    txt_files = find_all_txt_files(root_folder)
    files_found = len(txt_files)
    
    print(f"\nFound {files_found} files ending with 'features_summarized.txt'")
    print("=" * 80)
    
    # Process each file
    for idx, full_path in enumerate(txt_files, 1):
        filename = os.path.basename(full_path)
        dirpath = os.path.dirname(full_path)
        relative_path = os.path.relpath(dirpath, root_folder)
        
        print(f"\n[{idx}/{files_found}] Processing: {filename}")
        print(f"    Subfolder: {relative_path}")
        
        # Extract metadata from filename
        metadata = extract_metadata_from_filename(filename)
        
        # Read the txt file
        try:
            # Try to detect and load the file with different delimiters
            df = None
            delimiter_used = None
            
            # Try comma-separated first
            try:
                df = pd.read_csv(full_path, sep=',')
                if len(df.columns) > 1:  # Valid delimiter
                    delimiter_used = 'comma'
                else:
                    df = None
            except:
                pass
            
            # Try tab-separated
            if df is None:
                try:
                    df = pd.read_csv(full_path, sep='\t')
                    if len(df.columns) > 1:  # Valid delimiter
                        delimiter_used = 'tab'
                    else:
                        df = None
                except:
                    pass
            
            # Try whitespace-separated
            if df is None:
                try:
                    df = pd.read_csv(full_path, sep=r'\s+')
                    if len(df.columns) > 1:  # Valid delimiter
                        delimiter_used = 'whitespace'
                    else:
                        df = None
                except:
                    pass
            
            # If still None, try with automatic delimiter detection
            if df is None:
                try:
                    df = pd.read_csv(full_path, sep=None, engine='python')
                    delimiter_used = 'auto-detected'
                except:
                    pass
            
            if df is not None and len(df) > 0:
                # Add metadata columns
                df['group'] = metadata['group']
                df['mouse_id'] = metadata['mouse_id']
                df['section_number'] = metadata['section_number']
                df['position'] = metadata['position']
                df['series_number'] = metadata['series_number']
                df['filename'] = filename
                df['subfolder'] = relative_path
                df['subfolder_name'] = os.path.basename(dirpath)
                df['filepath'] = full_path
                
                all_data.append(df)
                files_loaded += 1
                print(f"    ✓ Loaded {len(df)} rows, {len(df.columns)-8} feature columns ({delimiter_used} delimiter)")
            else:
                # File is empty - create a single row with metadata and NaN for other fields
                print(f"    ✗ File is empty")
                df = pd.DataFrame([{
                    'group': metadata['group'],
                    'mouse_id': metadata['mouse_id'],
                    'section_number': metadata['section_number'],
                    'position': metadata['position'],
                    'series_number': metadata['series_number'],
                    'filename': filename,
                    'subfolder': relative_path,
                    'subfolder_name': os.path.basename(dirpath),
                    'filepath': full_path,
                    'load_error': 'Empty file'
                }])
                all_data.append(df)
            
        except Exception as e:
            print(f"    ✗ Error loading file: {str(e)}")
            
            # If can't load file, save metadata with NaN for other fields
            df = pd.DataFrame([{
                'group': metadata['group'],
                'mouse_id': metadata['mouse_id'],
                'section_number': metadata['section_number'],
                'position': metadata['position'],
                'series_number': metadata['series_number'],
                'filename': filename,
                'subfolder': relative_path,
                'subfolder_name': os.path.basename(dirpath),
                'filepath': full_path,
                'load_error': str(e)
            }])
            all_data.append(df)
    
    print("\n" + "=" * 80)
    print(f"SUMMARY:")
    print(f"  Total subdirectories searched: {len(all_subdirs)}")
    print(f"  Files found: {files_found}")
    print(f"  Files successfully loaded: {files_loaded}")
    print(f"  Files failed: {files_found - files_loaded}")
    
    # Combine all DataFrames
    if all_data:
        print(f"\nCombining {len(all_data)} DataFrames...")
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"✓ Combined DataFrame created with {len(combined_df)} total rows")
        print(f"  Total columns: {len(combined_df.columns)}")
        
        # Print column names
        print(f"\n  Column names:")
        for col in combined_df.columns:
            print(f"    - {col}")
        
        # Print summary statistics
        if 'group' in combined_df.columns and combined_df['group'].notna().any():
            print(f"\n  Data summary by group:")
            print(combined_df.groupby('group').size())
        
        if 'mouse_id' in combined_df.columns and combined_df['mouse_id'].notna().any():
            print(f"\n  Unique mice: {combined_df['mouse_id'].nunique()}")
        
        if 'subfolder_name' in combined_df.columns:
            print(f"\n  Files per subfolder:")
            print(combined_df.groupby('subfolder_name')['filename'].nunique())
        
        return combined_df
    else:
        print("\n✗ No files found or loaded!")
        return pd.DataFrame()

In [ ]:
# Set your root folder path
root_folder = r'AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/'

print("="*60)
print("Searching for features files...")
print("="*60)

# Load and combine all features files
df = find_and_load_features_files(root_folder)

In [ ]:
df.sample(10)

In [ ]:
df.shape

In [ ]:
# # Extract section number and rank within each mouse
# df['brain_region'] = df.groupby('mouse_id')['section_number'].transform(
#     lambda x: pd.cut(x.rank(method='dense'), bins=3, labels=['anterior_to_core', 'core', 'posterior_to_core'])
# )
# df.sample(5)

In [ ]:
# Add group column by matching mouse_id
mouse_list = pd.read_csv('mouse_list_microglia_morphology.csv', dtype=str)
mouse_list.sample(5)

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop(columns='load_error')
df_clean.shape

In [ ]:
df_clean.dropna().shape

In [ ]:
merged_df = pd.merge(df_clean, mouse_list, on='mouse_id')
merged_df.sample(5)

In [ ]:
# Save to CSV
output_file = 'AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/combined_features_data.csv'
merged_df.to_csv(output_file, index=False)
print(f"\n✓ Data saved to: {output_file}")

# Save as pickle for faster loading later
merged_df.to_pickle("AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/combined_features_data.pkl")
print(f"\n✓ Pickle file saved")

# Optionally save to Excel
# df.to_excel('combined_features_data.xlsx', index=False)

### Generate summary stats

In [ ]:
# 1) Number of mice per group
mice_per_group = merged_df.groupby('group_y')['mouse_id'].nunique().reset_index()
mice_per_group.columns = ['group', 'number_of_mice']
print("1) Number of mice per group:")
print(mice_per_group)
print("\n")

# 2) Number of sections per mouse
sections_per_mouse = merged_df.groupby(['group_y', 'mouse_id'])['section_number'].nunique().reset_index()
sections_per_mouse.columns = ['group', 'mouse_id', 'number_of_sections']
print("2) Number of sections per mouse:")
print(sections_per_mouse)
print("\n")

# 3) Number of positions per mouse
positions_per_mouse = merged_df.groupby(['group_y', 'mouse_id'])['position'].nunique().reset_index()
positions_per_mouse.columns = ['group', 'mouse_id', 'number_of_positions']
print("3) Number of sections per mouse:")
print(positions_per_mouse)
print("\n")

# 4) Total number of rows per mouse
rows_per_mouse = merged_df.groupby(['group_y', 'mouse_id']).size().reset_index()
rows_per_mouse.columns = ['group', 'mouse_id', 'total_rows']
print("4) Total number of microglia per mouse:")
print(rows_per_mouse)
print("\n")

# Optional: Combined summary per mouse
combined_summary = merged_df.groupby(['group_y', 'mouse_id']).agg(
    number_of_sections=('section_number', 'nunique'),
    number_of_positions=('position', 'nunique'),
    total_rows=('mouse_id', 'count')
).reset_index()
print("Combined summary per mouse:")
print(combined_summary)

### Visualize features (summarized)

In [ ]:
# Plot summary stats for each group (using a different color to differentiate mice within the same group)
def plot_feature_summary_by_mouse(df, feature_columns, group_column='group_y', 
                                   mouse_column='mouse_id',
                                   figsize_per_plot=(5, 4), cols_per_row=3,
                                   plot_type='box', save_path=None, group_order=None):
    """
    Plot summary statistics for each feature by group with different colors for each mouse.
    
    Args:
        df: DataFrame containing the data
        feature_columns: List of feature column names to plot
        group_column: Name of the column containing group labels
        mouse_column: Name of the column containing mouse IDs
        figsize_per_plot: Tuple of (width, height) for each subplot
        cols_per_row: Number of columns in the subplot grid
        plot_type: Type of plot ('box', 'violin', 'strip', 'swarm', or 'boxstrip')
        save_path: Optional path to save the figure
        group_order: Optional list specifying the order of groups (e.g., ['young', 'aged'])
                     If None, uses the order they appear in the data
    
    Returns:
        fig, axes: matplotlib figure and axes objects
    """
    
    # Calculate grid dimensions
    n_features = len(feature_columns)
    n_cols = min(cols_per_row, n_features)
    n_rows = int(np.ceil(n_features / n_cols))
    
    # Create figure
    fig_width = figsize_per_plot[0] * n_cols
    fig_height = figsize_per_plot[1] * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height))
    
    # Flatten axes array for easier iteration
    if n_features == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    # Set style
    sns.set_style("whitegrid")
    
    # Determine group order
    if group_order is None:
        groups = df[group_column].unique()
    else:
        # Validate that all specified groups exist in the data
        available_groups = set(df[group_column].unique())
        specified_groups = set(group_order)
        
        # Check for groups in order that don't exist in data
        missing_groups = specified_groups - available_groups
        if missing_groups:
            print(f"Warning: Groups {missing_groups} specified in group_order but not found in data")
        
        # Check for groups in data that aren't in the order
        extra_groups = available_groups - specified_groups
        if extra_groups:
            print(f"Warning: Groups {extra_groups} found in data but not in group_order. They will be appended.")
        
        # Use specified order for groups that exist, append any extras
        groups = [g for g in group_order if g in available_groups]
        groups.extend([g for g in df[group_column].unique() if g not in groups])
    
    print(f"Plotting groups in order: {groups}")
    
    # Create color mapping for mice within each group
    mouse_colors = {}
    
    # Define color palettes for each group position
    group_palettes = [
        "Blues",
        "Oranges", 
        "Greens",
        "Purples",
        "Reds",
        "YlOrBr"
    ]
    
    for group_idx, group in enumerate(groups):
        mice_in_group = df[df[group_column] == group][mouse_column].unique()
        n_mice = len(mice_in_group)
        
        # Select palette based on group index
        palette_name = group_palettes[group_idx % len(group_palettes)]
        palette = sns.color_palette(palette_name, n_mice + 2)[1:-1]  # Skip lightest and darkest
        
        for idx, mouse in enumerate(mice_in_group):
            mouse_colors[mouse] = palette[idx]
    
    # Plot each feature
    for idx, feature in enumerate(feature_columns):
        ax = axes[idx]
        
        # Remove rows with NaN in current feature, group, or mouse
        plot_data = df[[feature, group_column, mouse_column]].dropna()
        
        if len(plot_data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(feature)
            continue
        
        # Map colors to data points
        point_colors = plot_data[mouse_column].map(mouse_colors)
        
        # Create the plot based on type
        if plot_type == 'box':
            # Box plot with colored points by mouse
            sns.boxplot(data=plot_data, x=group_column, y=feature, ax=ax, 
                       order=groups, color='lightgray', width=0.5)
            sns.stripplot(data=plot_data, x=group_column, y=feature, ax=ax,
                         order=groups, hue=mouse_column, palette=mouse_colors, 
                         alpha=0.7, size=4, dodge=False, legend=False)
            
        elif plot_type == 'violin':
            # Violin plot with colored points by mouse
            sns.violinplot(data=plot_data, x=group_column, y=feature, ax=ax,
                          order=groups, color='lightgray', inner=None)
            sns.stripplot(data=plot_data, x=group_column, y=feature, ax=ax,
                         order=groups, hue=mouse_column, palette=mouse_colors,
                         alpha=0.7, size=4, dodge=False, legend=False)
            
        elif plot_type == 'strip':
            # Strip plot only with mouse colors
            sns.stripplot(data=plot_data, x=group_column, y=feature, ax=ax,
                         order=groups, hue=mouse_column, palette=mouse_colors,
                         alpha=0.7, size=5, dodge=False, legend=False)
            
        elif plot_type == 'swarm':
            # Swarm plot with mouse colors
            sns.swarmplot(data=plot_data, x=group_column, y=feature, ax=ax,
                         order=groups, hue=mouse_column, palette=mouse_colors,
                         alpha=0.7, size=4, dodge=False, legend=False)
            
        elif plot_type == 'boxstrip':
            # Box plot with strip overlay (default recommended)
            sns.boxplot(data=plot_data, x=group_column, y=feature, ax=ax,
                       order=groups, color='lightgray', width=0.6, linewidth=1.5,
                       showfliers=False)  # Don't show outliers to avoid duplication
            sns.stripplot(data=plot_data, x=group_column, y=feature, ax=ax,
                         order=groups, hue=mouse_column, palette=mouse_colors,
                         alpha=0.7, size=4, dodge=False, legend=False)
        
        # Formatting
        ax.set_title(feature, fontsize=11, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel(feature, fontsize=9)
        ax.tick_params(axis='x', rotation=45)
        
        # Add sample sizes in the specified order
        group_counts = plot_data.groupby(group_column).size()
        ax.set_xticklabels([f"{grp}\n(n={group_counts.get(grp, 0)})" 
                           for grp in groups])
    
    # Remove extra subplots
    for idx in range(n_features, len(axes)):
        fig.delaxes(axes[idx])
    
    # Create custom legend for mice (in specified group order)
    from matplotlib.patches import Patch
    legend_elements = []
    for group in groups:
        mice_in_group = sorted([m for m in mouse_colors.keys() 
                               if m in df[df[group_column] == group][mouse_column].unique()])
        if mice_in_group:
            legend_elements.append(Patch(facecolor='white', edgecolor='black', 
                                        label=f'{group}:'))
            for mouse in mice_in_group:
                legend_elements.append(Patch(facecolor=mouse_colors[mouse], 
                                            label=f'  Mouse {mouse}'))
    
    # Add legend to the figure
    fig.legend(handles=legend_elements, loc='center left', 
              bbox_to_anchor=(1.0, 0.5), fontsize=8, title='Groups & Mice')
    
    # Adjust layout
    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Make room for legend
    
    # Save if path provided
    if save_path:
        # Create directory if it doesn't exist
        save_dir = os.path.dirname(save_path)
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir, exist_ok=True)
            print(f"Created directory: {save_dir}")
        
        # Get the base path without extension
        base_path = os.path.splitext(save_path)[0]
        
        # Save in multiple formats
        formats = ['pdf', 'png', 'tiff']
        for fmt in formats:
            output_path = f"{base_path}.{fmt}"
            plt.savefig(output_path, dpi=300, bbox_inches='tight', format=fmt)
            print(f"Figure saved to: {output_path}")
    
    return fig, axes

In [ ]:
feature_list_sel = ['sphericity', 'circularity', 'volume_4', 'nodesN',
                   'nodesBranching','nodesEnding', 'branchNodesN', 'branchNodesPerBranch',
                   'branchEndNodesN', 'branchEndNodesPerBranch', 'branchSegmentsN', 'branchSegmentsPerBranch',
                   'branchCyclesN', 'branchLengthSkel_4', 'branchLengthRatio_3', 'closeness_4',
                   'betweenness_4']

In [ ]:
merged_df.columns

In [ ]:
merged_df.group_y.unique()

In [ ]:
group_plot_order = ['c57_young_male_ctrl', 'nsg_young_male_ctrl']

In [ ]:
plot_feature_summary_by_mouse(merged_df, feature_list_sel, group_column='group_y', 
                                   mouse_column='mouse_id',
                                   figsize_per_plot=(5, 4), cols_per_row=3,
                                   plot_type='box', save_path='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/plots/', group_order=group_plot_order)

### Microglia counts

Since the image size is the same across all sections, and there seems to be less microglia in NSG animals after the exclusion criteria, which is uniform across, we will start with the most basic analysis and look into the number of microglia between the two groups.

In [ ]:
def analyze_microglia_counts(df, group_column='group_y', output_folder='microglia_counts'):
    """
    Analyze and visualize the total number of microglia per section per mouse.
    Includes statistical comparisons between groups.
    
    Args:
        df: DataFrame with microglia data (one row per cell)
        group_column: Column containing group labels
        output_folder: Where to save outputs
        
    Returns:
        count_stats: DataFrame with statistical results
    """
    
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats
    from scipy.stats import mannwhitneyu, ttest_ind, shapiro
    from statannotations.Annotator import Annotator
    
    os.makedirs(output_folder, exist_ok=True)
    
    print("\n" + "="*80)
    print("ANALYZING MICROGLIA COUNTS PER SECTION")
    print("="*80 + "\n")
    
    # Filter for relevant groups
    df_filtered = df[df[group_column].isin(['c57_young_male_ctrl', 'nsg_young_male_ctrl'])].copy()
    
    # Ensure mouse_id is treated as string to handle alphanumeric IDs
    df_filtered['mouse_id'] = df_filtered['mouse_id'].astype(str)
    
    # Count microglia per mouse/section/series
    count_data = df_filtered.groupby([group_column, 'mouse_id', 'section_number', 'series_number']).size().reset_index(name='cell_count')
    
    # Add readable group labels
    count_data['Group'] = count_data[group_column].map({
        'c57_young_male_ctrl': 'C57',
        'nsg_young_male_ctrl': 'NSG'
    })
    
    print("Data summary:")
    print(f"  Total sections analyzed: {len(count_data)}")
    print(f"  C57 sections: {len(count_data[count_data['Group']=='C57'])}")
    print(f"  NSG sections: {len(count_data[count_data['Group']=='NSG'])}")
    print(f"  Total microglia counted: {count_data['cell_count'].sum()}")
    
    # Print unique mice per group
    print(f"\nUnique mice:")
    for group in ['C57', 'NSG']:
        mice = count_data[count_data['Group'] == group]['mouse_id'].unique()
        print(f"  {group}: {sorted(mice)}")
    
    # Save raw count data
    count_data.to_csv(os.path.join(output_folder, 'microglia_counts_per_section.csv'), index=False)
    print(f"\n✓ Saved: microglia_counts_per_section.csv")
    
    # Calculate statistics per mouse
    mouse_summary = count_data.groupby([group_column, 'Group', 'mouse_id']).agg({
        'cell_count': ['mean', 'std', 'sum', 'count']
    }).reset_index()
    
    mouse_summary.columns = [group_column, 'Group', 'mouse_id', 'mean_cells_per_section', 
                             'std_cells_per_section', 'total_cells', 'n_sections']
    
    mouse_summary.to_csv(os.path.join(output_folder, 'microglia_counts_per_mouse.csv'), index=False)
    print(f"✓ Saved: microglia_counts_per_mouse.csv")
    
    # Perform statistical tests
    c57_counts = count_data[count_data['Group'] == 'C57']['cell_count']
    nsg_counts = count_data[count_data['Group'] == 'NSG']['cell_count']
    
    # Descriptive statistics
    print("\n" + "-"*80)
    print("DESCRIPTIVE STATISTICS")
    print("-"*80)
    
    for group_name in ['C57', 'NSG']:
        group_counts = count_data[count_data['Group'] == group_name]['cell_count']
        print(f"\n{group_name}:")
        print(f"  N sections: {len(group_counts)}")
        print(f"  Mean ± SD: {group_counts.mean():.2f} ± {group_counts.std():.2f} cells/section")
        print(f"  Median [IQR]: {group_counts.median():.2f} [{group_counts.quantile(0.25):.2f}-{group_counts.quantile(0.75):.2f}]")
        print(f"  Range: {group_counts.min():.0f} - {group_counts.max():.0f}")
    
    # Test for normality
    print("\n" + "-"*80)
    print("NORMALITY TESTS (Shapiro-Wilk)")
    print("-"*80)
    
    _, p_shapiro_c57 = shapiro(c57_counts)
    _, p_shapiro_nsg = shapiro(nsg_counts)
    
    print(f"C57: p = {p_shapiro_c57:.4f} {'(Normal)' if p_shapiro_c57 > 0.05 else '(Not normal)'}")
    print(f"NSG: p = {p_shapiro_nsg:.4f} {'(Normal)' if p_shapiro_nsg > 0.05 else '(Not normal)'}")
    
    is_normal = (p_shapiro_c57 > 0.05) and (p_shapiro_nsg > 0.05)
    
    # Statistical comparison
    print("\n" + "-"*80)
    print("STATISTICAL COMPARISON")
    print("-"*80)
    
    # t-test (parametric)
    t_stat, p_ttest = ttest_ind(c57_counts, nsg_counts)
    
    # Mann-Whitney U (non-parametric)
    u_stat, p_mannwhitney = mannwhitneyu(c57_counts, nsg_counts, alternative='two-sided')
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((c57_counts.std()**2 + nsg_counts.std()**2) / 2)
    cohens_d = (nsg_counts.mean() - c57_counts.mean()) / pooled_std
    
    print(f"\nT-test: t = {t_stat:.4f}, p = {p_ttest:.4f}")
    print(f"Mann-Whitney U: U = {u_stat:.4f}, p = {p_mannwhitney:.4f}")
    print(f"Cohen's d: {cohens_d:.4f} ({'Small' if abs(cohens_d) < 0.5 else 'Medium' if abs(cohens_d) < 0.8 else 'Large'} effect)")
    
    # Choose appropriate test
    if is_normal:
        test_used = 't-test'
        p_value = p_ttest
        print(f"\nData is normally distributed → Using t-test")
    else:
        test_used = 'Mann-Whitney U'
        p_value = p_mannwhitney
        print(f"\nData is not normally distributed → Using Mann-Whitney U test")
    
    print(f"\n{'***SIGNIFICANT DIFFERENCE***' if p_value < 0.05 else 'No significant difference'} (p = {p_value:.4f})")
    
    # Store statistics
    stats_results = {
        'test_used': test_used,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'c57_mean': c57_counts.mean(),
        'c57_std': c57_counts.std(),
        'c57_median': c57_counts.median(),
        'nsg_mean': nsg_counts.mean(),
        'nsg_std': nsg_counts.std(),
        'nsg_median': nsg_counts.median()
    }
    
    # Create visualizations
    create_count_visualizations(count_data, mouse_summary, stats_results, output_folder)
    
    # Save statistics
    stats_df = pd.DataFrame([stats_results])
    stats_df.to_csv(os.path.join(output_folder, 'microglia_count_statistics.csv'), index=False)
    print(f"\n✓ Saved: microglia_count_statistics.csv")
    
    return count_data, mouse_summary, stats_results


def create_count_visualizations(count_data, mouse_summary, stats_results, output_folder):
    """Create comprehensive visualizations for microglia counts."""
    
    import matplotlib.pyplot as plt
    import seaborn as sns
    from statannotations.Annotator import Annotator
    import numpy as np
    
    colors = {'C57': '#4A90E2', 'NSG': '#50C878'}
    
    # Figure 1: Box plot per section with strip plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Plot 1: Per section counts
    ax = axes[0]
    
    # Create boxplot
    bp = ax.boxplot([count_data[count_data['Group'] == 'C57']['cell_count'],
                      count_data[count_data['Group'] == 'NSG']['cell_count']],
                     positions=[0, 1],
                     widths=0.5,
                     patch_artist=True,
                     showfliers=False)
    
    # Color the boxes
    bp['boxes'][0].set_facecolor(colors['C57'])
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(colors['NSG'])
    bp['boxes'][1].set_alpha(0.7)
    
    # Add strip plot overlay
    for idx, group in enumerate(['C57', 'NSG']):
        group_data = count_data[count_data['Group'] == group]['cell_count']
        x = np.random.normal(idx, 0.04, size=len(group_data))
        ax.scatter(x, group_data, alpha=0.5, s=30, color=colors[group],
                  edgecolor='black', linewidth=0.5)
    
    # Add statistical annotation manually
    y_max = count_data['cell_count'].max()
    y_range = count_data['cell_count'].max() - count_data['cell_count'].min()
    
    # Draw significance line
    x1, x2 = 0, 1
    y = y_max + y_range * 0.1
    h = y_range * 0.02
    
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='black')
    
    # Add p-value text
    if stats_results['p_value'] < 0.001:
        pvalue_text = "***\np < 0.001"
    elif stats_results['p_value'] < 0.01:
        pvalue_text = f"**\np = {stats_results['p_value']:.4f}"
    elif stats_results['p_value'] < 0.05:
        pvalue_text = f"*\np = {stats_results['p_value']:.4f}"
    else:
        pvalue_text = f"ns\np = {stats_results['p_value']:.4f}"
    
    ax.text((x1+x2)*.5, y+h, pvalue_text, ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['C57', 'NSG'])
    ax.set_xlabel('Group', fontsize=12, fontweight='bold')
    ax.set_ylabel('Microglia Count per Section', fontsize=12, fontweight='bold')
    ax.set_title(f'Microglia Counts per Section\n({stats_results["test_used"]})', 
                fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Plot 2: Per mouse averages
    ax = axes[1]
    
    # Separate data by group
    c57_mice = mouse_summary[mouse_summary['Group'] == 'C57']
    nsg_mice = mouse_summary[mouse_summary['Group'] == 'NSG']
    
    # Calculate means and stds for bar plot
    c57_mean = c57_mice['mean_cells_per_section'].mean()
    c57_std = c57_mice['mean_cells_per_section'].std()
    nsg_mean = nsg_mice['mean_cells_per_section'].mean()
    nsg_std = nsg_mice['mean_cells_per_section'].std()
    
    # Create bar plot
    x_pos = [0, 1]
    heights = [c57_mean, nsg_mean]
    errors = [c57_std, nsg_std]
    bar_colors = [colors['C57'], colors['NSG']]
    
    bars = ax.bar(x_pos, heights, color=bar_colors, alpha=0.7, 
                 edgecolor='black', linewidth=1.5, width=0.6,
                 yerr=errors, capsize=10, error_kw={'linewidth': 2})
    
    # Add individual mouse points
    for idx, group in enumerate(['C57', 'NSG']):
        group_data = mouse_summary[mouse_summary['Group'] == group]['mean_cells_per_section']
        x = np.random.normal(idx, 0.04, size=len(group_data))
        ax.scatter(x, group_data, alpha=0.8, s=80, color=colors[group],
                  edgecolor='black', linewidth=1, zorder=10)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['C57', 'NSG'])
    ax.set_xlabel('Group', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Microglia Count per Section', fontsize=12, fontweight='bold')
    ax.set_title('Average Counts per Mouse\n(Mean ± SD)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Plot 3: Distribution (violin plot)
    ax = axes[2]
    
    # Create violin plot using matplotlib
    parts = ax.violinplot([count_data[count_data['Group'] == 'C57']['cell_count'],
                           count_data[count_data['Group'] == 'NSG']['cell_count']],
                          positions=[0, 1],
                          widths=0.7,
                          showmeans=False,
                          showmedians=True)
    
    # Color the violins
    for idx, pc in enumerate(parts['bodies']):
        color = colors['C57'] if idx == 0 else colors['NSG']
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    
    # Color the median lines
    parts['cmedians'].set_edgecolor('black')
    parts['cmedians'].set_linewidth(2)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['C57', 'NSG'])
    ax.set_xlabel('Group', fontsize=12, fontweight='bold')
    ax.set_ylabel('Microglia Count per Section', fontsize=12, fontweight='bold')
    ax.set_title('Distribution of Counts\n(Violin Plot)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.suptitle('Microglia Cell Counts: C57 vs NSG', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, 'microglia_counts_comparison.pdf'), bbox_inches='tight', dpi=300)
    plt.savefig(os.path.join(output_folder, 'microglia_counts_comparison.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, 'microglia_counts_comparison.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: microglia_counts_comparison")
    
    # Figure 2: Per mouse breakdown
    create_per_mouse_breakdown(count_data, mouse_summary, colors, output_folder)
    
    # Figure 3: Per series comparison (if series data available)
    if 'series_number' in count_data.columns:
        create_per_series_comparison(count_data, colors, output_folder)


def create_per_mouse_breakdown(count_data, mouse_summary, colors, output_folder):
    """Create detailed per-mouse breakdown."""
    
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create grouped bar plot
    x_pos = []
    x_labels = []
    bar_heights = []
    bar_colors = []
    
    pos = 0
    for group in ['C57', 'NSG']:
        group_mice = mouse_summary[mouse_summary['Group'] == group].sort_values('mouse_id')
        
        for _, mouse in group_mice.iterrows():
            x_pos.append(pos)
            x_labels.append(f"{group}\n{mouse['mouse_id']}")  # Use mouse_id directly
            bar_heights.append(mouse['mean_cells_per_section'])
            bar_colors.append(colors[group])
            pos += 1
        
        pos += 0.5  # Add gap between groups
    
    bars = ax.bar(x_pos, bar_heights, color=bar_colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add error bars - FIXED VERSION
    for i, (x, height) in enumerate(zip(x_pos, bar_heights)):
        # Extract group and mouse_id from label
        group_label = x_labels[i].split('\n')[0]
        mouse_id_str = x_labels[i].split('\n')[1]  # Keep as string
        
        # Filter data using string comparison
        mouse_data = count_data[
            (count_data['Group'] == group_label) &
            (count_data['mouse_id'].astype(str) == mouse_id_str)  # Convert to string for comparison
        ]['cell_count']
        
        if len(mouse_data) > 1:
            std = mouse_data.std()
            ax.errorbar(x, height, yerr=std, fmt='none', ecolor='black', 
                       capsize=5, capthick=2, linewidth=2)
        
        # Add sample size on top of each bar
        n_sections = len(mouse_data)
        ax.text(x, height + (height * 0.02), f'n={n_sections}', 
               ha='center', va='bottom', fontsize=8)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9, rotation=0)
    ax.set_xlabel('Mouse ID', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Microglia Count per Section', fontsize=12, fontweight='bold')
    ax.set_title('Microglia Counts per Mouse (Mean ± SD)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors['C57'], alpha=0.7, edgecolor='black', label='C57'),
        Patch(facecolor=colors['NSG'], alpha=0.7, edgecolor='black', label='NSG')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, 'microglia_counts_per_mouse.pdf'), bbox_inches='tight', dpi=300)
    plt.savefig(os.path.join(output_folder, 'microglia_counts_per_mouse.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, 'microglia_counts_per_mouse.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: microglia_counts_per_mouse")


def create_per_series_comparison(count_data, colors, output_folder):
    """Create comparison of counts across different series (spatial locations)."""
    
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats
    import numpy as np
    import pandas as pd
    
    # Check if series_number exists and has multiple values
    if 'series_number' not in count_data.columns:
        return
    
    unique_series = count_data['series_number'].unique()
    if len(unique_series) <= 1:
        return
    
    # Map series numbers to anatomical locations
    series_mapping = {
        1: 'Core',
        2: '300-600μm',
        3: '600-900μm',
        4: 'Contralateral'
    }
    
    count_data['Location'] = count_data['series_number'].map(series_mapping)
    
    # Remove any unmapped locations
    count_data = count_data[count_data['Location'].notna()].copy()
    
    if len(count_data) == 0:
        print("No data available for series comparison")
        return
    
    # Calculate statistics per series
    series_stats = []
    
    for series_num in sorted(unique_series):
        location = series_mapping.get(series_num, f'Series {series_num}')
        
        c57_data = count_data[(count_data['Group'] == 'C57') & 
                              (count_data['series_number'] == series_num)]['cell_count']
        nsg_data = count_data[(count_data['Group'] == 'NSG') & 
                              (count_data['series_number'] == series_num)]['cell_count']
        
        if len(c57_data) > 0 and len(nsg_data) > 0:
            t_stat, p_val = stats.ttest_ind(c57_data, nsg_data)
            
            series_stats.append({
                'series': series_num,
                'location': location,
                'c57_mean': c57_data.mean(),
                'c57_std': c57_data.std(),
                'c57_n': len(c57_data),
                'nsg_mean': nsg_data.mean(),
                'nsg_std': nsg_data.std(),
                'nsg_n': len(nsg_data),
                'p_value': p_val,
                'significant': p_val < 0.05
            })
    
    if not series_stats:
        return
    
    series_stats_df = pd.DataFrame(series_stats)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Grouped box plot by series - USING MATPLOTLIB INSTEAD OF SEABORN
    ax = axes[0]
    
    locations = sorted(count_data['Location'].unique())
    
    # Prepare data for boxplot
    c57_data_by_location = []
    nsg_data_by_location = []
    
    for location in locations:
        c57_vals = count_data[(count_data['Group'] == 'C57') & 
                             (count_data['Location'] == location)]['cell_count'].values
        nsg_vals = count_data[(count_data['Group'] == 'NSG') & 
                             (count_data['Location'] == location)]['cell_count'].values
        c57_data_by_location.append(c57_vals)
        nsg_data_by_location.append(nsg_vals)
    
    # Create positions for boxes
    positions_c57 = np.arange(len(locations)) * 2 - 0.3
    positions_nsg = np.arange(len(locations)) * 2 + 0.3
    
    # Plot C57 boxes
    bp1 = ax.boxplot(c57_data_by_location, positions=positions_c57, widths=0.5,
                     patch_artist=True, showfliers=False,
                     boxprops=dict(facecolor=colors['C57'], alpha=0.7),
                     medianprops=dict(color='black', linewidth=2),
                     whiskerprops=dict(color='black'),
                     capprops=dict(color='black'))
    
    # Plot NSG boxes
    bp2 = ax.boxplot(nsg_data_by_location, positions=positions_nsg, widths=0.5,
                     patch_artist=True, showfliers=False,
                     boxprops=dict(facecolor=colors['NSG'], alpha=0.7),
                     medianprops=dict(color='black', linewidth=2),
                     whiskerprops=dict(color='black'),
                     capprops=dict(color='black'))
    
    # Add strip overlay
    for i, location in enumerate(locations):
        # C57 points
        c57_vals = count_data[(count_data['Group'] == 'C57') & 
                             (count_data['Location'] == location)]['cell_count'].values
        if len(c57_vals) > 0:
            x_c57 = np.random.normal(positions_c57[i], 0.04, size=len(c57_vals))
            ax.scatter(x_c57, c57_vals, alpha=0.5, s=30, color=colors['C57'],
                      edgecolor='black', linewidth=0.5)
        
        # NSG points
        nsg_vals = count_data[(count_data['Group'] == 'NSG') & 
                             (count_data['Location'] == location)]['cell_count'].values
        if len(nsg_vals) > 0:
            x_nsg = np.random.normal(positions_nsg[i], 0.04, size=len(nsg_vals))
            ax.scatter(x_nsg, nsg_vals, alpha=0.5, s=30, color=colors['NSG'],
                      edgecolor='black', linewidth=0.5)
    
    # Set x-axis
    ax.set_xticks(np.arange(len(locations)) * 2)
    ax.set_xticklabels(locations)
    ax.set_xlabel('Anatomical Location', fontsize=12, fontweight='bold')
    ax.set_ylabel('Microglia Count per Section', fontsize=12, fontweight='bold')
    ax.set_title('Microglia Counts by Anatomical Location', fontsize=13, fontweight='bold')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors['C57'], alpha=0.7, edgecolor='black', label='C57'),
        Patch(facecolor=colors['NSG'], alpha=0.7, edgecolor='black', label='NSG')
    ]
    ax.legend(handles=legend_elements, title='Group', fontsize=10, title_fontsize=11)
    
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Plot 2: Mean comparison by series
    ax = axes[1]
    
    x = np.arange(len(series_stats_df))
    width = 0.35
    
    c57_bars = ax.bar(x - width/2, series_stats_df['c57_mean'], width,
                     label='C57', color=colors['C57'], alpha=0.7,
                     edgecolor='black', linewidth=1.5,
                     yerr=series_stats_df['c57_std'], capsize=5,
                     error_kw={'linewidth': 2})
    
    nsg_bars = ax.bar(x + width/2, series_stats_df['nsg_mean'], width,
                     label='NSG', color=colors['NSG'], alpha=0.7,
                     edgecolor='black', linewidth=1.5,
                     yerr=series_stats_df['nsg_std'], capsize=5,
                     error_kw={'linewidth': 2})
    
    # Add significance markers
    max_height = max(series_stats_df['c57_mean'].max(), series_stats_df['nsg_mean'].max())
    max_std = max(series_stats_df['c57_std'].max(), series_stats_df['nsg_std'].max())
    
    for i, row in series_stats_df.iterrows():
        if row['significant']:
            y_pos = max_height + max_std + (max_height * 0.1)
            ax.plot([i - width/2, i + width/2], [y_pos, y_pos], 'k-', linewidth=1.5)
            
            if row['p_value'] < 0.001:
                sig_text = '***'
            elif row['p_value'] < 0.01:
                sig_text = '**'
            else:
                sig_text = '*'
            
            ax.text(i, y_pos + (max_height * 0.02), sig_text, ha='center', fontsize=12, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(series_stats_df['location'])
    ax.set_xlabel('Anatomical Location', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean Microglia Count per Section', fontsize=12, fontweight='bold')
    ax.set_title('Mean Counts by Location (±SD)\n* p<0.05, ** p<0.01, *** p<0.001', 
                fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.suptitle('Spatial Distribution of Microglia Counts', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, 'microglia_counts_by_series.pdf'), bbox_inches='tight', dpi=300)
    plt.savefig(os.path.join(output_folder, 'microglia_counts_by_series.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, 'microglia_counts_by_series.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: microglia_counts_by_series")
    
    # Save series statistics
    series_stats_df.to_csv(os.path.join(output_folder, 'microglia_counts_by_series_stats.csv'), index=False)
    print(f"✓ Saved: microglia_counts_by_series_stats.csv")
    
    # Print series statistics
    print("\n" + "-"*80)
    print("COUNTS BY ANATOMICAL LOCATION")
    print("-"*80)
    for _, row in series_stats_df.iterrows():
        print(f"\n{row['location']}:")
        print(f"  C57: {row['c57_mean']:.2f} ± {row['c57_std']:.2f} (n={row['c57_n']})")
        print(f"  NSG: {row['nsg_mean']:.2f} ± {row['nsg_std']:.2f} (n={row['nsg_n']})")
        print(f"  p-value: {row['p_value']:.4f} {'***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else 'ns'}")
        

In [ ]:
# Analyze microglia counts
count_data, mouse_summary, stats_results = analyze_microglia_counts(
    df=merged_df,
    group_column='group_y',
    output_folder='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/microglia_counts_analysis'
)

print("\n" + "="*80)
print("MICROGLIA COUNT ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  • microglia_counts_per_section.csv - Raw counts per section")
print("  • microglia_counts_per_mouse.csv - Summary statistics per mouse")
print("  • microglia_count_statistics.csv - Statistical test results")
print("  • microglia_counts_comparison.pdf/png/tiff - Main comparison plots")
print("  • microglia_counts_per_mouse.pdf/png/tiff - Per-mouse breakdown")
print("  • microglia_counts_by_series.pdf/png/tiff - Spatial distribution (if applicable)")
print("\n")

### Unsupervised feature selection
Since there are so many selected features, we will start identifying features with the most differences between the two groups.

In the paper, "[to] reduce the dimensionality of the feature set, we performed a principle component analysis (PCA) with centering and scaling of the features. Only features which had an AUC score above 0.85 were included in the PCA. If a feature (e.g., volume of the skeleton nodes) was determined with different percentiles (e.g., median over all nodes, or 75th percentile, see above), we included only the variant with the highest AUC score in the PCA."

For this PCA, since we are using features already narrowed down by the paper, we are doing further selection on the features.

**Variance threshold** removes features (columns) that have low variance across samples, meaning the feature values don't change much between samples. Here, we are removing features with variance <0.1.
* 0.0 - removes only constant features (all values identical)
* 0.01-0.1 - removes very low variance features (common choice)
* Higher values are more aggressive

**Correlation threshold** removes highly correlated features (redundant features); if two features are highly correlated (e.g., correlation > 0.95), they provide similar information.
* 0.90-0.95 - aggressive removal (common choice)
* 0.95-0.99 - moderate removal
* 0.99+ - only removes nearly identical features

**Keep_top_n_variants** keeps only the top N features with the highest variance across samples and selects the most variable features before applying PCA. 

In [ ]:
def select_features_unsupervised(df, feature_list, 
                                variance_threshold=0.1,
                                correlation_threshold=0.90,
                                keep_top_n_variants=1,
                                save_path=None):
    """
    Select features without using target variable (fully unsupervised).
    - Remove low variance features
    - Among similar feature variants (e.g., _median, _p75), keep only specified number
    - Remove highly correlated features (keep one from each correlated pair)
    
    Args:
        df: DataFrame with features
        feature_list: List of feature names to consider
        variance_threshold: Minimum variance to keep feature
        correlation_threshold: Maximum correlation to keep both features
        keep_top_n_variants: Number of variants to keep for similar features (default=1)
    
    Returns:
        final_features: List of selected feature names
    """
    import re
    
    print("="*60)
    print("UNSUPERVISED FEATURE SELECTION")
    print("="*60)
    
    # Step 1: Remove features not in DataFrame
    print("\nStep 1: Checking feature availability...")
    available_features = [f for f in feature_list if f in df.columns]
    print(f"  Available features: {len(available_features)}/{len(feature_list)}")
    
    # Step 2: Remove low variance features
    print(f"\nStep 2: Filtering by variance > {variance_threshold}...")
    X = df[available_features].copy()
    variances = X.var()
    high_var_features = variances[variances > variance_threshold].index.tolist()
    print(f"  Features after variance filter: {len(high_var_features)}")
    
    # Step 3: Group similar features and keep top N variants
    print(f"\nStep 3: Selecting top {keep_top_n_variants} variant(s) for similar features...")
    
    feature_groups = {}
    
    for feature in high_var_features:
        # Extract base feature name (remove percentile suffixes)
        # Patterns: _median, _mean, _std, _min, _max, _p25, _p50, _p75, _p90, etc.
        base_name = re.sub(r'_(median|mean|std|min|max|p\d+)$', '', feature)
        
        if base_name not in feature_groups:
            feature_groups[base_name] = []
        
        feature_groups[base_name].append(feature)
    
    # For each group, keep variants based on their variance (higher variance = more informative)
    selected_variants = []
    
    for base_name, variants in feature_groups.items():
        if len(variants) > 1:
            # Sort by variance (descending)
            variant_vars = [(v, variances[v]) for v in variants]
            variant_vars.sort(key=lambda x: x[1], reverse=True)
            
            # Keep top N
            keep_variants = [v[0] for v in variant_vars[:keep_top_n_variants]]
            selected_variants.extend(keep_variants)
            
            print(f"\n  {base_name}: {len(variants)} variants found")
            for i, (var, variance) in enumerate(variant_vars[:3]):  # Show top 3
                marker = "✓" if var in keep_variants else " "
                print(f"    {marker} {var}: variance = {variance:.4f}")
        else:
            # Only one variant, keep it
            selected_variants.extend(variants)
    
    print(f"\n  After variant selection: {len(selected_variants)} features")
    
    # Step 4: Remove highly correlated features
    print(f"\nStep 4: Removing highly correlated features (|r| > {correlation_threshold})...")
    
    X_filtered = df[selected_variants].copy()
    corr_matrix = X_filtered.corr().abs()
    
    # Find pairs of highly correlated features
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    
    # For each highly correlated pair, keep the one with higher variance
    to_drop = set()
    
    for i, col in enumerate(upper_tri.columns):
        for j, row in enumerate(upper_tri.columns):
            if i < j and upper_tri.iloc[i, j] > correlation_threshold:
                # Get variance for both features
                var_col = variances.get(col, 0)
                var_row = variances.get(row, 0)
                
                # Drop the one with lower variance
                if var_col < var_row:
                    to_drop.add(col)
                    print(f"  Dropping {col} (var={var_col:.4f}) - correlated with {row} (var={var_row:.4f}, r={upper_tri.iloc[i, j]:.3f})")
                else:
                    to_drop.add(row)
                    print(f"  Dropping {row} (var={var_row:.4f}) - correlated with {col} (var={var_col:.4f}, r={upper_tri.iloc[i, j]:.3f})")
    
    final_features = [f for f in selected_variants if f not in to_drop]
    
    print(f"\n  Removed {len(to_drop)} highly correlated features")
    print(f"  Final feature count: {len(final_features)}")
    
    # Summary
    print("\n" + "="*60)
    print(f"FINAL SELECTED FEATURES: {len(final_features)}")
    print("="*60)
    
    # Sort by variance (most variable features first)
    final_features_sorted = sorted(final_features, 
                                   key=lambda x: variances[x], 
                                   reverse=True)
    
    print("\nTop 20 features by variance:")
    for i, feat in enumerate(final_features_sorted[:20], 1):
        print(f"{i:3d}. {feat:50s} variance = {variances[feat]:.4f}")
    
    if len(final_features_sorted) > 20:
        print(f"\n... and {len(final_features_sorted) - 20} more features")

    # Save selected features if path provided
    if save_path:
        # Create directory if it doesn't exist
        save_dir = os.path.dirname(save_path)
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir, exist_ok=True)
            print(f"\nCreated directory: {save_dir}")
        
        # Determine file format and save
        file_ext = os.path.splitext(save_path)[1].lower()
        
        if file_ext == '.csv':
            # Save as CSV with variance information
            import pandas as pd
            feature_df = pd.DataFrame({
                'feature': final_features_sorted,
                'variance': [variances[f] for f in final_features_sorted]
            })
            feature_df.to_csv(save_path, index=False)
            print(f"\nSelected features saved to: {save_path} (CSV format)")
        else:
            # Save as plain text (one feature per line)
            with open(save_path, 'w') as f:
                for feat in final_features_sorted:
                    f.write(f"{feat}\n")
            print(f"\nSelected features saved to: {save_path} (TXT format)")
    
    return final_features_sorted

In [ ]:
selected_features = select_features_unsupervised(
    df=merged_df,
    feature_list=feature_list_sel,
    variance_threshold=0.1,
    correlation_threshold=0.90,
    keep_top_n_variants=1,
    save_path='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/plots/pca/selected_features.csv'
)

### PCA
Now that we have the list of features selected, we will apply PCA to the dataset. Since these are control brains and there is no core, plotting the results based on "zones (i.e., distance from core) will not provide any information. We will only plot the results based on group (i.e., c57_male_ctrl and nsg_male_ctrl).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

def compute_pca(df, selected_features, 
                group_column1='series_number',
                group_column2=None,  # Changed default to None
                n_components=10):
    """
    Compute PCA with centering and scaling (purely unsupervised).
    This function runs the PCA computation only once.
    
    Args:
        df: DataFrame with features
        selected_features: List of features to include in PCA
        group_column1: First grouping column (e.g., 'series_number'). Set to None if not needed.
        group_column2: Second grouping column (e.g., 'group_y'). Set to None if not needed.
        n_components: Number of principal components to compute
        
    Returns:
        pca_results: Dictionary with all PCA results and metadata
    """
    
    print("\n" + "="*80)
    print("COMPUTING PCA (UNSUPERVISED)")
    print("="*80)
    
    # Prepare data
    X = df[selected_features].copy()
    
    # Get grouping variables if columns exist
    group1_exists = group_column1 is not None and group_column1 in df.columns
    group2_exists = group_column2 is not None and group_column2 in df.columns
    
    if group_column1 is not None:
        if group1_exists:
            group1 = df[group_column1].copy()
            print(f"\nGrouping variable 1: {group_column1}")
        else:
            group1 = None
            print(f"\nWarning: {group_column1} not found in dataframe")
    else:
        group1 = None
        group1_exists = False
        print("\nGrouping variable 1: None (not specified)")
    
    if group_column2 is not None:
        if group2_exists:
            group2 = df[group_column2].copy()
            print(f"Grouping variable 2: {group_column2}")
        else:
            group2 = None
            print(f"Warning: {group_column2} not found in dataframe")
    else:
        group2 = None
        group2_exists = False
        print("Grouping variable 2: None (not specified)")
    
    # Remove rows with any NaN values in features
    mask = ~X.isna().any(axis=1)
    X_clean = X[mask].copy()
    
    # Also remove NaN from grouping variables
    if group1_exists:
        group1_clean = group1[mask].copy()
        mask_g1 = group1_clean.notna()
        X_clean = X_clean[mask_g1]
        group1_clean = group1_clean[mask_g1]
    else:
        group1_clean = None
    
    if group2_exists:
        group2_clean = group2[mask].copy()
        if group1_exists:
            group2_clean = group2_clean[X_clean.index]
        mask_g2 = group2_clean.notna()
        X_clean = X_clean[mask_g2]
        if group1_exists:
            group1_clean = group1_clean[mask_g2]
        group2_clean = group2_clean[mask_g2]
    else:
        group2_clean = None
    
    print(f"\nData shape: {X_clean.shape}")
    print(f"Samples: {len(X_clean)}")
    print(f"Features: {len(selected_features)}")
    
    if group1_exists:
        print(f"\n{group_column1} distribution:")
        print(group1_clean.value_counts().sort_index())
    
    if group2_exists:
        print(f"\n{group_column2} distribution:")
        print(group2_clean.value_counts())
    
    # Standardize features (centering and scaling)
    print("\nStandardizing features (centering and scaling)...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)
    
    # Perform PCA (completely unsupervised)
    print(f"Performing PCA with {n_components} components...")
    pca = PCA(n_components=min(n_components, len(selected_features), len(X_clean)))
    X_pca = pca.fit_transform(X_scaled)
    
    # Create DataFrame with PC scores
    pc_columns = [f'PC{i+1}' for i in range(X_pca.shape[1])]
    pca_df = pd.DataFrame(X_pca, columns=pc_columns, index=X_clean.index)
    
    if group1_exists:
        pca_df[group_column1] = group1_clean.values
    if group2_exists:
        pca_df[group_column2] = group2_clean.values
    
    # Print explained variance
    print("\n" + "-"*80)
    print("EXPLAINED VARIANCE")
    print("-"*80)
    for i, var in enumerate(pca.explained_variance_ratio_[:5], 1):
        print(f"PC{i}: {var*100:.2f}%")
    print(f"\nCumulative variance (PC1-PC5): {pca.explained_variance_ratio_[:5].sum()*100:.2f}%")
    
    # Calculate loadings
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    loadings_df = pd.DataFrame(
        pca.components_.T,
        columns=[f'PC{i+1}' for i in range(pca.components_.shape[0])],
        index=selected_features
    )
    
    # Create explained variance dataframe
    variance_df = pd.DataFrame({
        'PC': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
        'Explained_Variance_Ratio': pca.explained_variance_ratio_,
        'Explained_Variance': pca.explained_variance_,
        'Cumulative_Variance_Ratio': np.cumsum(pca.explained_variance_ratio_)
    })
    
    # Create color mappings
    palettes = create_color_palettes(pca_df, group_column1, group_column2, 
                                     group1_exists, group2_exists)
    
    print("\n" + "="*80)
    print("PCA COMPUTATION COMPLETE")
    print("="*80)
    
    # Return all results
    return {
        'pca': pca,
        'scaler': scaler,
        'pca_df': pca_df,
        'X_scaled': X_scaled,
        'X_clean': X_clean,
        'X_pca': X_pca,
        'group1': group1_clean,
        'group2': group2_clean,
        'group_column1': group_column1,
        'group_column2': group_column2,
        'group1_exists': group1_exists,
        'group2_exists': group2_exists,
        'selected_features': selected_features,
        'loadings': loadings,
        'loadings_df': loadings_df,
        'explained_variance': variance_df,
        'palettes': palettes,
        'n_components': X_pca.shape[1]
    }


In [ ]:
def create_color_palettes(pca_df, group_column1, group_column2, group1_exists, group2_exists):
    """Create color palettes and label mappings for both grouping variables."""
    
    palettes = {}
    
    # Palette for group_column1 (series_number)
    if group1_exists and group_column1 in pca_df.columns:
        if group_column1 == 'series_number':
            series_mapping = {
                1: 'core', 2: '300-600um', 3: '600-900um', 4: 'contralateral',
                '1': 'core', '2': '300-600um', '3': '600-900um', '4': 'contralateral'
            }
            pca_df[f'{group_column1}_label'] = pca_df[group_column1].map(series_mapping)
            
            palettes['group1'] = {
                'column': f'{group_column1}_label',
                'palette': {
                    'core': '#E74C3C',
                    '300-600um': '#F39C12',
                    '600-900um': '#3498DB',
                    'contralateral': '#2ECC71'
                },
                'original_column': group_column1
            }
        else:
            palettes['group1'] = {
                'column': group_column1,
                'palette': None,
                'original_column': group_column1
            }
    
    # Palette for group_column2 (group_y)
    if group2_exists and group_column2 in pca_df.columns:
        if 'group' in group_column2.lower():
            group_labels = {
                'c57_young_male_ctrl': 'C57 4m.o. Male Control',
                'nsg_young_male_ctrl': 'NSG 4m.o. Male Control'
            }
            pca_df[f'{group_column2}_label'] = pca_df[group_column2].map(group_labels)
            
            palettes['group2'] = {
                'column': f'{group_column2}_label',
                'palette': {
                    'c57_young_male_ctrl': '#4A90E2',  # Blue for C57
                    'nsg_young_male_ctrl': '#50C878'    # Green for NSG
                },
                'original_column': group_column2,
                'label_map': group_labels
            }
        else:
            palettes['group2'] = {
                'column': group_column2,
                'palette': None,
                'original_column': group_column2
            }
    
    return palettes

In [ ]:
def plot_explained_variance(pca_results, output_folder):
    """Plot explained variance."""
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca = pca_results['pca']
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(range(1, len(pca.explained_variance_ratio_)+1), 
           pca.explained_variance_ratio_ * 100,
           alpha=0.7, color='steelblue')
    ax.plot(range(1, len(pca.explained_variance_ratio_)+1),
            np.cumsum(pca.explained_variance_ratio_) * 100,
            'r-o', linewidth=2, markersize=8, label='Cumulative')
    ax.set_xlabel('Principal Component', fontsize=12, fontweight='bold')
    ax.set_ylabel('Explained Variance (%)', fontsize=12, fontweight='bold')
    ax.set_title('PCA Explained Variance (Unsupervised)', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, 'pca_explained_variance.pdf'))
    plt.savefig(os.path.join(output_folder, 'pca_explained_variance.png'), dpi=300)
    plt.savefig(os.path.join(output_folder, 'pca_explained_variance.tiff'), dpi=300)
    plt.close()
    
    print(f"✓ Saved: pca_explained_variance")

In [ ]:
def plot_pc_scatter(pca_results, output_folder, pc_x=1, pc_y=2, group_num=1):
    """
    Plot scatter plot of two PCs colored by specified group.
    
    Args:
        pca_results: Results from compute_pca()
        output_folder: Where to save plots
        pc_x: PC number for x-axis (1-indexed)
        pc_y: PC number for y-axis (1-indexed)
        group_num: Which grouping to use (1 or 2)
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca_df = pca_results['pca_df']
    pca = pca_results['pca']
    palettes = pca_results['palettes']
    
    # Check if PCs exist
    if pc_x > pca_results['n_components'] or pc_y > pca_results['n_components']:
        print(f"Warning: Requested PCs not available. Only {pca_results['n_components']} components computed.")
        return
    
    # Select group
    group_key = f'group{group_num}'
    if group_key not in palettes:
        print(f"Warning: Group {group_num} not available. Available groups: {list(palettes.keys())}")
        return
    
    color_column = palettes[group_key]['column']
    palette = palettes[group_key]['palette']
    group_column = palettes[group_key]['original_column']
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    for group in pca_df[color_column].unique():
        mask = pca_df[color_column] == group
        color = palette.get(group) if palette else None
        ax.scatter(pca_df.loc[mask, f'PC{pc_x}'], 
                  pca_df.loc[mask, f'PC{pc_y}'],
                  label=group,
                  s=100, alpha=0.7, edgecolors='black', linewidth=0.5,
                  color=color)
    
    ax.set_xlabel(f'PC{pc_x} ({pca.explained_variance_ratio_[pc_x-1]*100:.2f}%)', 
                 fontsize=13, fontweight='bold')
    ax.set_ylabel(f'PC{pc_y} ({pca.explained_variance_ratio_[pc_y-1]*100:.2f}%)', 
                 fontsize=13, fontweight='bold')
    ax.set_title(f'PCA: PC{pc_x} vs PC{pc_y}\nColored by {group_column}', 
                fontsize=15, fontweight='bold')
    ax.legend(title=group_column, fontsize=10, title_fontsize=11, 
             framealpha=0.9, edgecolor='black')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    
    filename = f'pca_PC{pc_x}_PC{pc_y}_by_{group_column}'
    plt.savefig(os.path.join(output_folder, f'{filename}.pdf'))
    plt.savefig(os.path.join(output_folder, f'{filename}.png'), dpi=300)
    plt.savefig(os.path.join(output_folder, f'{filename}.tiff'), dpi=300)
    plt.close()
    
    print(f"✓ Saved: {filename}")

In [ ]:
def plot_pc_combined(pca_results, output_folder, pc_x=1, pc_y=2):
    """Plot side-by-side comparison of both groupings."""
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca_df = pca_results['pca_df']
    pca = pca_results['pca']
    palettes = pca_results['palettes']
    
    if 'group1' not in palettes or 'group2' not in palettes:
        print("Warning: Both groups needed for combined plot")
        print(f"Available groups: {list(palettes.keys())}")
        return
    
    # Check if PCs exist
    if pc_x > pca_results['n_components'] or pc_y > pca_results['n_components']:
        print(f"Warning: Requested PCs not available. Only {pca_results['n_components']} components computed.")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Left plot: group 1
    ax = axes[0]
    color_column1 = palettes['group1']['column']
    palette1 = palettes['group1']['palette']
    group_column1 = palettes['group1']['original_column']
    
    for group in pca_df[color_column1].unique():
        mask = pca_df[color_column1] == group
        color = palette1.get(group) if palette1 else None
        ax.scatter(pca_df.loc[mask, f'PC{pc_x}'], 
                  pca_df.loc[mask, f'PC{pc_y}'],
                  label=group,
                  s=100, alpha=0.7, edgecolors='black', linewidth=0.5,
                  color=color)
    
    ax.set_xlabel(f'PC{pc_x} ({pca.explained_variance_ratio_[pc_x-1]*100:.2f}%)', 
                 fontsize=12, fontweight='bold')
    ax.set_ylabel(f'PC{pc_y} ({pca.explained_variance_ratio_[pc_y-1]*100:.2f}%)', 
                 fontsize=12, fontweight='bold')
    ax.set_title(f'Colored by {group_column1}', fontsize=14, fontweight='bold')
    ax.legend(title=group_column1, fontsize=9, title_fontsize=10, 
             framealpha=0.9, edgecolor='black')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Right plot: group 2
    ax = axes[1]
    color_column2 = palettes['group2']['column']
    palette2 = palettes['group2']['palette']
    group_column2 = palettes['group2']['original_column']
    
    for group in pca_df[color_column2].unique():
        mask = pca_df[color_column2] == group
        if palette2:
            # Get the color from palette using the display label
            color = palette2.get(group)
            # If color not found by label, try getting original group name
            if color is None and 'label_map' in palettes['group2']:
                # Get original group name from pca_df
                orig_group = pca_df.loc[mask, group_column2].iloc[0]
                color = palette2.get(orig_group)
        else:
            color = None
        
        ax.scatter(pca_df.loc[mask, f'PC{pc_x}'], 
                  pca_df.loc[mask, f'PC{pc_y}'],
                  label=group,
                  s=100, alpha=0.7, edgecolors='black', linewidth=0.5,
                  color=color)
    
    ax.set_xlabel(f'PC{pc_x} ({pca.explained_variance_ratio_[pc_x-1]*100:.2f}%)', 
                 fontsize=12, fontweight='bold')
    ax.set_ylabel(f'PC{pc_y} ({pca.explained_variance_ratio_[pc_y-1]*100:.2f}%)', 
                 fontsize=12, fontweight='bold')
    ax.set_title(f'Colored by {group_column2}', fontsize=14, fontweight='bold')
    ax.legend(title=group_column2, fontsize=9, title_fontsize=10, 
             framealpha=0.9, edgecolor='black')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.suptitle(f'PCA: PC{pc_x} vs PC{pc_y} - Dual Grouping Visualization', 
                fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    filename = f'pca_PC{pc_x}_PC{pc_y}_combined'
    plt.savefig(os.path.join(output_folder, f'{filename}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {filename}")

In [ ]:
def plot_3d_pca(pca_results, output_folder, group_num=1):
    """
    Create 3D scatter plot of PC1, PC2, PC3.
    
    Args:
        pca_results: Results from compute_pca()
        output_folder: Where to save plots
        group_num: Which grouping to use (1 or 2)
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    if pca_results['n_components'] < 3:
        print("Warning: Need at least 3 PCs for 3D plot")
        return
    
    pca_df = pca_results['pca_df']
    pca = pca_results['pca']
    palettes = pca_results['palettes']
    
    # Select group
    group_key = f'group{group_num}'
    if group_key not in palettes:
        print(f"Warning: Group {group_num} not available")
        return
    
    color_column = palettes[group_key]['column']
    palette = palettes[group_key]['palette']
    group_column = palettes[group_key]['original_column']
    
    # Create 3D plot
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    for group in pca_df[color_column].unique():
        mask = pca_df[color_column] == group
        color = palette.get(group) if palette else None
        ax.scatter(pca_df.loc[mask, 'PC1'], 
                  pca_df.loc[mask, 'PC2'],
                  pca_df.loc[mask, 'PC3'],
                  label=group,
                  s=100, alpha=0.7, edgecolors='black', linewidth=0.5,
                  color=color)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)', 
                 fontsize=11, fontweight='bold')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)', 
                 fontsize=11, fontweight='bold')
    ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)', 
                 fontsize=11, fontweight='bold')
    ax.set_title(f'PCA 3D Plot\nColored by {group_column}', 
                fontsize=14, fontweight='bold', pad=20)
    ax.legend(title=group_column, fontsize=9, title_fontsize=10, 
             framealpha=0.9, edgecolor='black')
    ax.grid(True, alpha=0.3)
    
    filename = f'pca_3D_by_{group_column}'
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{filename}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {filename}")

In [ ]:
def plot_feature_loadings(pca_results, output_folder, pc_x=1, pc_y=2, top_n=20):
    """
    Plot feature loadings as arrows on PC space.
    
    Args:
        pca_results: Results from compute_pca()
        output_folder: Where to save plots
        pc_x: PC number for x-axis (1-indexed)
        pc_y: PC number for y-axis (1-indexed)
        top_n: Number of top features to label (None = all)
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca = pca_results['pca']
    loadings = pca_results['loadings']
    selected_features = pca_results['selected_features']
    
    # Check if PCs exist
    if pc_x > pca_results['n_components'] or pc_y > pca_results['n_components']:
        print(f"Warning: Requested PCs not available")
        return
    
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Get loadings for specified PCs
    pc_x_idx = pc_x - 1
    pc_y_idx = pc_y - 1
    
    # Calculate magnitude of loadings for filtering
    magnitudes = np.sqrt(loadings[:, pc_x_idx]**2 + loadings[:, pc_y_idx]**2)
    
    # Get top N features if specified
    if top_n is not None:
        top_indices = np.argsort(magnitudes)[-top_n:]
    else:
        top_indices = range(len(selected_features))
    
    # Plot all loading vectors
    for i in range(len(selected_features)):
        ax.arrow(0, 0, loadings[i, pc_x_idx], loadings[i, pc_y_idx],
                head_width=0.02, head_length=0.02, 
                fc='steelblue', ec='steelblue', alpha=0.3, linewidth=0.5)
    
    # Label top N features
    for i in top_indices:
        feature = selected_features[i]
        ax.arrow(0, 0, loadings[i, pc_x_idx], loadings[i, pc_y_idx],
                head_width=0.03, head_length=0.03, 
                fc='darkblue', ec='darkblue', alpha=0.8, linewidth=1.5)
        ax.text(loadings[i, pc_x_idx]*1.15, loadings[i, pc_y_idx]*1.15, feature,
               fontsize=8, ha='center', va='center',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                        alpha=0.8, edgecolor='gray'))
    
    ax.set_xlabel(f'PC{pc_x} Loadings ({pca.explained_variance_ratio_[pc_x_idx]*100:.2f}%)', 
                 fontsize=13, fontweight='bold')
    ax.set_ylabel(f'PC{pc_y} Loadings ({pca.explained_variance_ratio_[pc_y_idx]*100:.2f}%)', 
                 fontsize=13, fontweight='bold')
    
    title = f'PCA Feature Loadings (PC{pc_x} vs PC{pc_y})'
    if top_n is not None:
        title += f'\nTop {top_n} features labeled'
    ax.set_title(title, fontsize=15, fontweight='bold')
    
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='k', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Set equal aspect ratio
    max_loading = np.abs(loadings[:, [pc_x_idx, pc_y_idx]]).max()
    ax.set_xlim(-max_loading*1.3, max_loading*1.3)
    ax.set_ylim(-max_loading*1.3, max_loading*1.3)
    ax.set_aspect('equal')
    
    filename = f'pca_loadings_PC{pc_x}_PC{pc_y}'
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{filename}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {filename}")

In [ ]:
def plot_top_features_barplot(pca_results, output_folder, pc_num=1, top_n=15):
    """
    Plot top contributing features for a specific PC as horizontal bar plot.
    
    Args:
        pca_results: Results from compute_pca()
        output_folder: Where to save plots
        pc_num: Which PC to analyze (1-indexed)
        top_n: Number of top features to show
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca = pca_results['pca']
    selected_features = pca_results['selected_features']
    
    if pc_num > pca_results['n_components']:
        print(f"Warning: PC{pc_num} not available")
        return
    
    pc_idx = pc_num - 1
    pc_loadings = pca.components_[pc_idx]
    
    # Get top N features by absolute loading value
    top_idx = np.argsort(np.abs(pc_loadings))[-top_n:]
    top_features = [selected_features[i] for i in top_idx]
    top_values = pc_loadings[top_idx]
    
    # Create plot
    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
    
    colors = ['#3498DB' if x > 0 else '#E74C3C' for x in top_values]
    y_pos = np.arange(len(top_features))
    
    ax.barh(y_pos, top_values, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features, fontsize=10)
    ax.set_xlabel('Loading Value', fontsize=12, fontweight='bold')
    ax.set_title(f'Top {top_n} Features Contributing to PC{pc_num}\n({pca.explained_variance_ratio_[pc_idx]*100:.2f}% variance explained)', 
                fontsize=13, fontweight='bold')
    ax.axvline(x=0, color='k', linestyle='-', linewidth=1.5)
    ax.grid(True, alpha=0.3, axis='x')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3498DB', alpha=0.7, edgecolor='black', label='Positive loading'),
        Patch(facecolor='#E74C3C', alpha=0.7, edgecolor='black', label='Negative loading')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    
    filename = f'pca_top{top_n}_features_PC{pc_num}'
    plt.savefig(os.path.join(output_folder, f'{filename}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'{filename}.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {filename}")

In [ ]:
def save_pca_results(pca_results, output_folder):
    """
    Save PCA scores, loadings, and variance to CSV files.
    
    Args:
        pca_results: Results from compute_pca()
        output_folder: Where to save CSV files
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    print("\nSaving PCA results to CSV files...")
    
    # Save PCA scores
    pca_scores_file = os.path.join(output_folder, 'pca_scores.csv')
    pca_results['pca_df'].to_csv(pca_scores_file, index=True)
    print(f"✓ PCA scores saved to: pca_scores.csv")
    
    # Save loadings
    loadings_file = os.path.join(output_folder, 'pca_loadings.csv')
    pca_results['loadings_df'].to_csv(loadings_file)
    print(f"✓ PCA loadings saved to: pca_loadings.csv")
    
    # Save explained variance
    variance_file = os.path.join(output_folder, 'pca_explained_variance.csv')
    pca_results['explained_variance'].to_csv(variance_file, index=False)
    print(f"✓ Explained variance saved to: pca_explained_variance.csv")
    
    print(f"\nAll results saved to: {output_folder}")

In [ ]:
def plot_all_pca_visualizations(pca_results, output_folder):
    """
    Generate all PCA visualizations at once.
    
    Args:
        pca_results: Results from compute_pca()
        output_folder: Where to save all plots
    """
    
    print("\n" + "="*80)
    print("GENERATING ALL PCA VISUALIZATIONS")
    print("="*80)
    
    # 1. Explained variance
    plot_explained_variance(pca_results, output_folder)
    
    # 2. PC scatter plots for each group
    if pca_results['group1_exists']:
        plot_pc_scatter(pca_results, output_folder, pc_x=1, pc_y=2, group_num=1)
        if pca_results['n_components'] >= 3:
            plot_pc_scatter(pca_results, output_folder, pc_x=1, pc_y=3, group_num=1)
            plot_pc_scatter(pca_results, output_folder, pc_x=2, pc_y=3, group_num=1)
        plot_3d_pca(pca_results, output_folder, group_num=1)
    
    if pca_results['group2_exists']:
        plot_pc_scatter(pca_results, output_folder, pc_x=1, pc_y=2, group_num=2)
        if pca_results['n_components'] >= 3:
            plot_pc_scatter(pca_results, output_folder, pc_x=1, pc_y=3, group_num=2)
            plot_pc_scatter(pca_results, output_folder, pc_x=2, pc_y=3, group_num=2)
        plot_3d_pca(pca_results, output_folder, group_num=2)
    
    # 3. Combined plots (if both groups exist)
    if pca_results['group1_exists'] and pca_results['group2_exists']:
        plot_pc_combined(pca_results, output_folder, pc_x=1, pc_y=2)
        if pca_results['n_components'] >= 3:
            plot_pc_combined(pca_results, output_folder, pc_x=1, pc_y=3)
            plot_pc_combined(pca_results, output_folder, pc_x=2, pc_y=3)
    
    # 4. Feature loadings
    plot_feature_loadings(pca_results, output_folder, pc_x=1, pc_y=2, top_n=20)
    if pca_results['n_components'] >= 3:
        plot_feature_loadings(pca_results, output_folder, pc_x=1, pc_y=3, top_n=20)
    
    # 5. Top features bar plots
    plot_top_features_barplot(pca_results, output_folder, pc_num=1, top_n=15)
    plot_top_features_barplot(pca_results, output_folder, pc_num=2, top_n=15)
    if pca_results['n_components'] >= 3:
        plot_top_features_barplot(pca_results, output_folder, pc_num=3, top_n=15)
    
    # 6. Save CSV files
    save_pca_results(pca_results, output_folder)
    
    print("\n" + "="*80)
    print("ALL VISUALIZATIONS COMPLETE")
    print("="*80)
    print(f"All files saved to: {output_folder}\n")

In [ ]:
# Step 1: Compute PCA once
pca_results = compute_pca(
    df=merged_df,
    selected_features=selected_features,
    group_column1='group_y',
    group_column2=None,
    n_components=10
)

# Step 2: Generate all plots at once
plot_all_pca_visualizations(pca_results, output_folder='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/plots/pca')

# Step 3: Access the results directly if needed
pca_df = pca_results['pca_df']  # PC scores with group labels
loadings_df = pca_results['loadings_df']  # Feature loadings
variance_df = pca_results['explained_variance']  # Variance explained

### Visualize results in the original data

In [ ]:
def visualize_top_pc1_features_with_stats(pca_results, df, output_folder, top_n=10, 
                                          alpha=0.05, correction_method='fdr_bh'):
    """
    Visualize the top features contributing to PC1 in the original data,
    comparing between the two groups with comprehensive statistics.
    Only shows statistical annotations for significant results.
    
    Args:
        pca_results: Results from compute_pca()
        df: Original dataframe with features and group labels
        output_folder: Where to save plots
        top_n: Number of top features to visualize
        alpha: Significance level (default 0.05)
        correction_method: Multiple testing correction ('bonferroni', 'fdr_bh', etc.)
    """
    
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats
    from scipy.stats import mannwhitneyu, shapiro, levene
    from statannotations.Annotator import Annotator
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca = pca_results['pca']
    selected_features = pca_results['selected_features']
    group_column = pca_results['group_column1']
    
    # Get top N features by PC1 loading (absolute value)
    pc1_loadings = pca.components_[0]
    top_idx = np.argsort(np.abs(pc1_loadings))[-top_n:][::-1]  # Descending order
    top_features = [selected_features[i] for i in top_idx]
    top_loadings = pc1_loadings[top_idx]
    
    # Filter data for valid rows (same as used in PCA)
    mask = ~df[selected_features].isna().any(axis=1)
    mask = mask & df[group_column].notna()
    df_clean = df[mask].copy()
    
    # Filter for only the two groups
    df_groups = df_clean[df_clean[group_column].isin(['c57_young_male_ctrl', 'nsg_young_male_ctrl'])]
    
    # Add readable group labels
    df_groups['Group_Label'] = df_groups[group_column].map({
        'c57_young_male_ctrl': 'C57',
        'nsg_young_male_ctrl': 'NSG'
    })
    
    print(f"\n{'='*80}")
    print(f"STATISTICAL ANALYSIS: Top {top_n} Features from PC1")
    print(f"{'='*80}\n")
    print(f"Sample sizes:")
    print(f"  C57: {len(df_groups[df_groups[group_column]=='c57_young_male_ctrl'])}")
    print(f"  NSG: {len(df_groups[df_groups[group_column]=='nsg_young_male_ctrl'])}")
    print(f"\nSignificance level: {alpha}")
    print(f"Multiple testing correction: {correction_method}\n")
    
    # Collect statistics for all features
    stats_results = []
    
    for feature in top_features:
        c57_data = df_groups[df_groups[group_column] == 'c57_young_male_ctrl'][feature].dropna()
        nsg_data = df_groups[df_groups[group_column] == 'nsg_young_male_ctrl'][feature].dropna()
        
        if len(c57_data) > 0 and len(nsg_data) > 0:
            # Descriptive statistics
            c57_mean = c57_data.mean()
            c57_std = c57_data.std()
            c57_median = c57_data.median()
            c57_sem = stats.sem(c57_data)
            
            nsg_mean = nsg_data.mean()
            nsg_std = nsg_data.std()
            nsg_median = nsg_data.median()
            nsg_sem = stats.sem(nsg_data)
            
            # Effect size (Cohen's d)
            pooled_std = np.sqrt((c57_std**2 + nsg_std**2) / 2)
            cohens_d = (nsg_mean - c57_mean) / pooled_std if pooled_std > 0 else np.nan
            
            # Fold change
            fold_change = nsg_mean / c57_mean if c57_mean != 0 else np.nan
            
            # Test for normality (Shapiro-Wilk)
            if len(c57_data) >= 3:
                _, p_shapiro_c57 = shapiro(c57_data)
            else:
                p_shapiro_c57 = np.nan
                
            if len(nsg_data) >= 3:
                _, p_shapiro_nsg = shapiro(nsg_data)
            else:
                p_shapiro_nsg = np.nan
            
            # Test for equal variances (Levene's test)
            if len(c57_data) > 0 and len(nsg_data) > 0:
                _, p_levene = levene(c57_data, nsg_data)
            else:
                p_levene = np.nan
            
            # Determine normality
            is_normal = (p_shapiro_c57 > 0.05 and p_shapiro_nsg > 0.05) if not np.isnan(p_shapiro_c57) else False
            
            # Parametric test (t-test) - always compute
            t_stat, p_ttest = stats.ttest_ind(c57_data, nsg_data, equal_var=(p_levene > 0.05))
            
            # Non-parametric test (Mann-Whitney U)
            u_stat, p_mannwhitney = mannwhitneyu(c57_data, nsg_data, alternative='two-sided')
            
            # Choose appropriate test
            if is_normal and p_levene > 0.05:
                test_used = 't-test'
                p_value = p_ttest
            else:
                test_used = 'Mann-Whitney U'
                p_value = p_mannwhitney
            
            stats_results.append({
                'feature': feature,
                'pc1_loading': pc1_loadings[selected_features.index(feature)],
                'c57_n': len(c57_data),
                'c57_mean': c57_mean,
                'c57_std': c57_std,
                'c57_sem': c57_sem,
                'c57_median': c57_median,
                'nsg_n': len(nsg_data),
                'nsg_mean': nsg_mean,
                'nsg_std': nsg_std,
                'nsg_sem': nsg_sem,
                'nsg_median': nsg_median,
                'fold_change': fold_change,
                'cohens_d': cohens_d,
                'p_shapiro_c57': p_shapiro_c57,
                'p_shapiro_nsg': p_shapiro_nsg,
                'is_normal': is_normal,
                'p_levene': p_levene,
                'test_used': test_used,
                'p_value_raw': p_value,
                'p_ttest': p_ttest,
                'p_mannwhitney': p_mannwhitney
            })
    
    # Convert to DataFrame
    stats_df = pd.DataFrame(stats_results)
    
    # Apply multiple testing correction
    from statsmodels.stats.multitest import multipletests
    
    if len(stats_df) > 0:
        reject, p_corrected, _, _ = multipletests(stats_df['p_value_raw'], 
                                                   alpha=alpha, 
                                                   method=correction_method)
        stats_df['p_value_corrected'] = p_corrected
        stats_df['significant'] = reject
        stats_df['significance_level'] = stats_df['p_value_corrected'].apply(
            lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        )
    
    # Print summary table
    print(f"{'Feature':<40} {'Test':<15} {'p-value':<10} {'p-adj':<10} {'Sig':<5} {'Effect Size (d)':<15}")
    print("-" * 105)
    for _, row in stats_df.iterrows():
        print(f"{row['feature']:<40} {row['test_used']:<15} {row['p_value_raw']:<10.4f} "
              f"{row['p_value_corrected']:<10.4f} {row['significance_level']:<5} {row['cohens_d']:<15.3f}")
    
    print(f"\n{sum(stats_df['significant'])} out of {len(stats_df)} features are significant after {correction_method} correction")
    
    # Save detailed statistics
    stats_df.to_csv(os.path.join(output_folder, 'top_PC1_features_detailed_statistics.csv'), index=False)
    print(f"\n✓ Saved: top_PC1_features_detailed_statistics.csv")
    
    # Create subplots
    n_cols = 2
    n_rows = int(np.ceil(top_n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5*n_rows))
    axes = axes.flatten()
    
    colors = {'C57': '#4A90E2', 'NSG': '#50C878'}
    
    for idx, feature in enumerate(top_features):
        ax = axes[idx]
        
        # Prepare data for this feature
        plot_data = df_groups[['Group_Label', feature]].copy()
        plot_data.columns = ['Group', 'Value']
        plot_data = plot_data.dropna()
        
        # Create box plot with seaborn
        sns.boxplot(data=plot_data, x='Group', y='Value', 
                   palette=colors, ax=ax, width=0.5,
                   order=['C57', 'NSG'])
        
        # Add strip plot
        sns.stripplot(data=plot_data, x='Group', y='Value',
                     palette=colors, ax=ax, alpha=0.6, size=5,
                     edgecolor='black', linewidth=0.5,
                     order=['C57', 'NSG'])
        
        # Get statistics for this feature
        feature_stats = stats_df[stats_df['feature'] == feature].iloc[0]
        
        # Only add statistical annotation if significant
        if feature_stats['significant']:
            pairs = [('C57', 'NSG')]
            
            annotator = Annotator(ax, pairs, data=plot_data, 
                                 x='Group', y='Value', order=['C57', 'NSG'])
            
            # Format p-value display
            if feature_stats['p_value_corrected'] < 0.001:
                pvalue_format = f"p < 0.001"
            else:
                pvalue_format = f"p = {feature_stats['p_value_corrected']:.4f}"
            
            annotator.set_custom_annotations([pvalue_format])
            annotator.configure(text_format='simple', 
                              loc='inside',
                              fontsize=9)
            annotator.annotate()
        
        # Formatting
        loading = feature_stats['pc1_loading']
        cohens_d = feature_stats['cohens_d']
        sig_marker = feature_stats['significance_level']
        
        ax.set_xlabel('')
        ax.set_ylabel(feature, fontsize=9)
        
        title_parts = [
            f'{feature}',
            f'PC1 loading: {loading:.3f} | d: {cohens_d:.3f}'
        ]
        
        if not feature_stats['significant']:
            title_parts.append('(ns)')
        
        ax.set_title('\n'.join(title_parts), fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Remove extra subplots
    for idx in range(top_n, len(axes)):
        fig.delaxes(axes[idx])
    
    sig_count = sum(stats_df['significant'])
    plt.suptitle(f'Top {top_n} Features Contributing to PC1 - C57 vs NSG\n'
                f'{sig_count}/{top_n} significant after {correction_method} correction (α={alpha})', 
                fontsize=14, fontweight='bold', y=0.998)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, f'top{top_n}_PC1_features_comparison_stats.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'top{top_n}_PC1_features_comparison_stats.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'top{top_n}_PC1_features_comparison_stats.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: top{top_n}_PC1_features_comparison_stats")

    # Create summary plot: only significant features
    sig_features = stats_df[stats_df['significant']]['feature'].tolist()
    
    if len(sig_features) > 0:
        n_sig = len(sig_features)
        n_cols_sig = 2
        n_rows_sig = int(np.ceil(n_sig / n_cols_sig))
        fig2, axes2 = plt.subplots(n_rows_sig, n_cols_sig, figsize=(14, 5*n_rows_sig))
        
        if n_sig == 1:
            axes2 = [axes2]
        else:
            axes2 = axes2.flatten()
        
        for idx, feature in enumerate(sig_features):
            ax = axes2[idx]
            
            # Prepare data
            plot_data = df_groups[['Group_Label', feature]].copy()
            plot_data.columns = ['Group', 'Value']
            plot_data = plot_data.dropna()
            
            # Create plots
            sns.boxplot(data=plot_data, x='Group', y='Value', 
                       palette=colors, ax=ax, width=0.5,
                       order=['C57', 'NSG'])
            
            sns.stripplot(data=plot_data, x='Group', y='Value',
                         palette=colors, ax=ax, alpha=0.6, size=5,
                         edgecolor='black', linewidth=0.5,
                         order=['C57', 'NSG'])
            
            # Get statistics
            feature_stats = stats_df[stats_df['feature'] == feature].iloc[0]
            
            # Add annotation
            pairs = [('C57', 'NSG')]
            annotator = Annotator(ax, pairs, data=plot_data, 
                                 x='Group', y='Value', order=['C57', 'NSG'])
            
            if feature_stats['p_value_corrected'] < 0.001:
                pvalue_format = f"***\np < 0.001"
            elif feature_stats['p_value_corrected'] < 0.01:
                pvalue_format = f"**\np = {feature_stats['p_value_corrected']:.4f}"
            else:
                pvalue_format = f"*\np = {feature_stats['p_value_corrected']:.4f}"
            
            annotator.set_custom_annotations([pvalue_format])
            annotator.configure(text_format='simple', 
                              loc='inside',
                              fontsize=10)
            annotator.annotate()
            
            # Formatting
            loading = feature_stats['pc1_loading']
            cohens_d = feature_stats['cohens_d']
            
            ax.set_xlabel('')
            ax.set_ylabel(feature, fontsize=9)
            ax.set_title(f'{feature}\nPC1 loading: {loading:.3f} | Cohen\'s d: {cohens_d:.3f}', 
                        fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        
        # Remove extra subplots
        for idx in range(n_sig, len(axes2)):
            fig2.delaxes(axes2[idx])
        
        plt.suptitle(f'Significant Features from PC1 - C57 vs NSG\n'
                    f'{n_sig} features significant after {correction_method} correction (α={alpha})', 
                    fontsize=14, fontweight='bold', y=0.998)
        plt.tight_layout()
        
        plt.savefig(os.path.join(output_folder, f'significant_PC1_features_only.pdf'), bbox_inches='tight')
        plt.savefig(os.path.join(output_folder, f'significant_PC1_features_only.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(output_folder, f'significant_PC1_features_only.tiff'), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Saved: significant_PC1_features_only ({n_sig} features)")
    else:
        print("\nNo significant features found after correction.")
    
    return stats_df

In [ ]:
def plot_effect_sizes(pca_results, df, output_folder, top_n=20, min_effect_size=0.0):
    """
    Create a dot plot showing effect sizes (Cohen's d) for top features.
    
    Args:
        pca_results: Results from compute_pca()
        df: Original dataframe
        output_folder: Where to save plots
        top_n: Number of top features to show
        min_effect_size: Minimum absolute effect size to display
    """
    
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import stats
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca = pca_results['pca']
    selected_features = pca_results['selected_features']
    group_column = pca_results['group_column1']
    
    # Get top N features by PC1 loading
    pc1_loadings = pca.components_[0]
    top_idx = np.argsort(np.abs(pc1_loadings))[-top_n:][::-1]
    top_features = [selected_features[i] for i in top_idx]
    
    # Filter data
    mask = ~df[selected_features].isna().any(axis=1)
    mask = mask & df[group_column].notna()
    df_clean = df[mask].copy()
    df_groups = df_clean[df_clean[group_column].isin(['c57_young_male_ctrl', 'nsg_young_male_ctrl'])]
    
    # Calculate effect sizes
    effect_sizes = []
    for feature in top_features:
        c57_data = df_groups[df_groups[group_column] == 'c57_young_male_ctrl'][feature].dropna()
        nsg_data = df_groups[df_groups[group_column] == 'nsg_young_male_ctrl'][feature].dropna()
        
        if len(c57_data) > 0 and len(nsg_data) > 0:
            c57_mean = c57_data.mean()
            c57_std = c57_data.std()
            nsg_mean = nsg_data.mean()
            nsg_std = nsg_data.std()
            
            pooled_std = np.sqrt((c57_std**2 + nsg_std**2) / 2)
            cohens_d = (nsg_mean - c57_mean) / pooled_std if pooled_std > 0 else 0
            
            # T-test
            t_stat, p_val = stats.ttest_ind(c57_data, nsg_data)
            
            effect_sizes.append({
                'feature': feature,
                'cohens_d': cohens_d,
                'abs_cohens_d': abs(cohens_d),
                'p_value': p_val,
                'significant': p_val < 0.05,
                'pc1_loading': pc1_loadings[selected_features.index(feature)]
            })
    
    effect_df = pd.DataFrame(effect_sizes)
    
    # Filter by minimum effect size
    effect_df = effect_df[effect_df['abs_cohens_d'] >= min_effect_size]
    
    # Sort by absolute effect size
    effect_df = effect_df.sort_values('abs_cohens_d', ascending=True)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(10, max(8, len(effect_df)*0.4)))
    
    # Color by significance and direction
    colors = []
    for _, row in effect_df.iterrows():
        if row['significant']:
            if row['cohens_d'] > 0:
                colors.append('#50C878')  # NSG higher (green)
            else:
                colors.append('#4A90E2')  # C57 higher (blue)
        else:
            colors.append('gray')
    
    y_pos = np.arange(len(effect_df))
    ax.barh(y_pos, effect_df['cohens_d'], color=colors, alpha=0.7, edgecolor='black', linewidth=1)
    
    # Add vertical line at 0
    ax.axvline(x=0, color='black', linestyle='-', linewidth=2)
    
    # Add effect size interpretation lines
    ax.axvline(x=0.2, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=-0.2, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=-0.5, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0.8, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(x=-0.8, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(effect_df['feature'], fontsize=9)
    ax.set_xlabel("Cohen's d (Effect Size)", fontsize=12, fontweight='bold')
    ax.set_title(f"Effect Sizes for Top {top_n} Features (PC1)\nC57 vs NSG", 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add text annotations for effect size magnitude
    ax.text(0.2, -0.5, 'Small', ha='center', va='top', fontsize=8, color='gray', transform=ax.get_xaxis_transform())
    ax.text(0.5, -0.5, 'Medium', ha='center', va='top', fontsize=8, color='gray', transform=ax.get_xaxis_transform())
    ax.text(0.8, -0.5, 'Large', ha='center', va='top', fontsize=8, color='gray', transform=ax.get_xaxis_transform())
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#4A90E2', alpha=0.7, edgecolor='black', label='C57 > NSG (p<0.05)'),
        Patch(facecolor='#50C878', alpha=0.7, edgecolor='black', label='NSG > C57 (p<0.05)'),
        Patch(facecolor='gray', alpha=0.7, edgecolor='black', label='Not significant')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_folder, f'effect_sizes_PC1_top{top_n}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'effect_sizes_PC1_top{top_n}.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(output_folder, f'effect_sizes_PC1_top{top_n}.tiff'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: effect_sizes_PC1_top{top_n}")
    
    # Save effect sizes to CSV
    effect_df.to_csv(os.path.join(output_folder, 'effect_sizes_PC1_features.csv'), index=False)
    print(f"✓ Saved: effect_sizes_PC1_features.csv")
    
    return effect_df

In [ ]:
# Step 4: Visualize differences in original data with statistics
stats_df = visualize_top_pc1_features_with_stats(
    pca_results=pca_results,
    df=merged_df,
    output_folder='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/plots/pca/feature_comparisons',
    top_n=15,
    alpha=0.05,
    correction_method='fdr_bh'  # or 'bonferroni'
)

# Step 5: Plot effect sizes
effect_df = plot_effect_sizes(
    pca_results=pca_results,
    df=merged_df,
    output_folder='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/plots/pca/feature_comparisons',
    top_n=20,
    min_effect_size=0.0
)

# Step 6: Print summary
print("\n" + "="*80)
print("SUMMARY OF SIGNIFICANT DIFFERENCES")
print("="*80)
print(f"\nFeatures with significant differences (FDR-corrected):")
sig_features = stats_df[stats_df['significant']]
for _, row in sig_features.iterrows():
    direction = "NSG > C57" if row['cohens_d'] > 0 else "C57 > NSG"
    print(f"  • {row['feature']:<40} {direction:>12} (d={row['cohens_d']:>6.3f}, p={row['p_value_corrected']:.4f})")

print(f"\nTotal: {len(sig_features)}/{len(stats_df)} features significantly different")

### Find representative samples

In [ ]:
def find_representative_samples(pca_results, df, output_folder='representative_samples'):
    """
    Find the most representative samples (closest to group centroid) for each group
    in PCA space. Returns mouse_id, section_number, and series_number.
    
    Args:
        pca_results: Results from compute_pca()
        df: Original dataframe with metadata
        output_folder: Where to save results
        
    Returns:
        representative_samples: DataFrame with representative samples for each group
    """
    
    import os
    import numpy as np
    import pandas as pd
    from scipy.spatial.distance import cdist
    
    os.makedirs(output_folder, exist_ok=True)
    
    pca_df = pca_results['pca_df']
    group_column = pca_results['group_column1']
    
    print("\n" + "="*80)
    print("FINDING MOST REPRESENTATIVE SAMPLES FOR EACH GROUP")
    print("="*80 + "\n")
    
    # Get PC columns
    pc_cols = [col for col in pca_df.columns if col.startswith('PC')]
    
    # Merge with original metadata
    metadata_cols = ['mouse_id', 'section_number', 'series_number', 'filename', 'filepath']
    available_metadata = [col for col in metadata_cols if col in df.columns]
    
    merged_data = pca_df.merge(
        df[available_metadata].reset_index(),
        left_index=True,
        right_on='index',
        how='left'
    )
    
    representative_samples = []
    
    for group_name in ['c57_young_male_ctrl', 'nsg_young_male_ctrl']:
        group_data = merged_data[merged_data[group_column] == group_name]
        
        if len(group_data) == 0:
            print(f"Warning: No samples found for {group_name}")
            continue
        
        print(f"\n{group_name.upper()}")
        print("-" * 80)
        print(f"Total samples: {len(group_data)}")
        
        # Calculate centroid in PCA space (using first 3 PCs)
        centroid = group_data[pc_cols[:3]].mean().values
        
        # Calculate distance of each sample to centroid
        distances = cdist(
            group_data[pc_cols[:3]].values,
            centroid.reshape(1, -1),
            metric='euclidean'
        ).flatten()
        
        # Add distances to dataframe
        group_data_copy = group_data.copy()
        group_data_copy['distance_to_centroid'] = distances
        
        # Sort by distance (closest first)
        group_data_sorted = group_data_copy.sort_values('distance_to_centroid')
        
        # Get top 10 most representative
        top_representative = group_data_sorted.head(10)
        
        print(f"\nCentroid coordinates (PC1, PC2, PC3): [{centroid[0]:.3f}, {centroid[1]:.3f}, {centroid[2]:.3f}]")
        print(f"\nTop 10 most representative samples:")
        print("-" * 80)
        print(f"{'Rank':<6} {'Mouse ID':<12} {'Section':<10} {'Series':<8} {'Distance':<12} {'PC1':<10} {'PC2':<10} {'PC3':<10}")
        print("-" * 80)
        
        for idx, (_, row) in enumerate(top_representative.iterrows(), 1):
            mouse_id = row.get('mouse_id', 'N/A')
            section = row.get('section_number', 'N/A')
            series = row.get('series_number', 'N/A')
            distance = row['distance_to_centroid']
            pc1 = row['PC1']
            pc2 = row['PC2']
            pc3 = row['PC3']
            filename = row.get('filename', 'N/A')
            filepath = row.get('filepath', 'N/A')
            
            print(f"{idx:<6} {str(mouse_id):<12} {str(section):<10} {str(series):<8} {distance:<12.4f} {pc1:<10.3f} {pc2:<10.3f} {pc3:<10.3f}")
            
            # Store for output
            representative_samples.append({
                'group': group_name,
                'rank': idx,
                'mouse_id': mouse_id,
                'section_number': section,
                'series_number': series,
                'distance_to_centroid': distance,
                'PC1': pc1,
                'PC2': pc2,
                'PC3': pc3,
                'filename': filename,
                'filepath': filepath
            })
        
        # Get #1 most representative
        most_rep = top_representative.iloc[0]
        print(f"\n>>> MOST REPRESENTATIVE SAMPLE:")
        print(f"    Mouse ID: {most_rep.get('mouse_id', 'N/A')}")
        print(f"    Section: {most_rep.get('section_number', 'N/A')}")
        print(f"    Series: {most_rep.get('series_number', 'N/A')}")
        print(f"    Distance to centroid: {most_rep['distance_to_centroid']:.4f}")
        print(f"    File: {most_rep.get('filename', 'N/A')}")
        if 'filepath' in most_rep:
            print(f"    Path: {most_rep.get('filepath', 'N/A')}")
    
    # Convert to DataFrame
    representative_df = pd.DataFrame(representative_samples)
    
    # Save to CSV
    csv_path = os.path.join(output_folder, 'representative_samples.csv')
    representative_df.to_csv(csv_path, index=False)
    print(f"\n{'='*80}")
    print(f"✓ Saved: {csv_path}")
    print(f"{'='*80}\n")
    
    return representative_df

In [ ]:
# Find representative samples
representative_df = find_representative_samples(
    pca_results=pca_results,
    df=merged_df,
    output_folder='AnalyzedData/Sherlock_NSG_4m_male_C57_4m_male/plots/representative_samples'
)

# Quick access to #1 most representative for each group
print("\n" + "="*80)
print("SUMMARY: #1 MOST REPRESENTATIVE SAMPLE PER GROUP")
print("="*80)

for group_name in ['c57_young_male_ctrl', 'nsg_young_male_ctrl']:
    top_sample = representative_df[
        (representative_df['group'] == group_name) & 
        (representative_df['rank'] == 1)
    ].iloc[0]
    
    group_label = 'C57' if 'c57' in group_name else 'NSG'
    print(f"\n{group_label}:")
    print(f"  Mouse ID: {top_sample['mouse_id']}")
    print(f"  Section: {top_sample['section_number']}")
    print(f"  Series: {top_sample['series_number']}")
    print(f"  File: {top_sample['filename']}")